# 04 · Robustness, errors and ablations
`metrics.json` (robustness), `errors.json` and `ablations.json` (Δ vs the full system with a paired group bootstrap, 90 % interval).

In [1]:
import json
from pathlib import Path

import pandas as pd

R = Path("../reports")  # the only directory read (scripts/run_notebooks.py --check-reads-only)


def load(name):
    return json.loads((R / name).read_text(encoding="utf-8"))
m = load("metrics.json")
print(json.dumps(m.get("robustness", {}), indent=1)[:1500])

{
 "note": "data/robustness/probes.csv does not exist yet (P2.5 is a human task); per-style scores appear once it is written",
 "status": "skipped"
}


In [2]:
err = load("errors.json")
e = pd.DataFrame(err["our_test"])
if len(e):
    e["gold_cat"] = e.gold.str.split(" / ").str[0]
    e["pred_cat"] = e.pred.str.split(" / ").str[0]
    display(e.groupby(["gold_cat", "pred_cat", "lang"]).size().sort_values(ascending=False).head(15))

gold_cat          pred_cat          lang   
Transportation    Transportation    arabizi    8
Finance           Registration      ar         7
Academic Support  Registration      ar         6
Transportation    Transportation    en         5
Academic Support  IT Support        en         4
Transportation    Transportation    ar         4
Facilities        Facilities        ar         4
Registration      Registration      en         3
Finance           Registration      arabizi    2
Transportation    Facilities        ar         2
Finance           Registration      en         2
Facilities        Facilities        en         2
Academic Support  Academic Support  ar         1
                  Registration      arabizi    1
                  Academic Support  en         1
dtype: int64

In [3]:
abl = load("ablations.json")
rows = []
for arm, a in abl["arms"].items():
    for target in ("category", "priority"):
        o = a.get("oof_official", {}).get(target, {})
        t = a.get("our_test", {}).get(target, {})
        rows.append({"arm": arm, "target": target, "oof": o.get("macro_f1"), "oof_delta": o.get("delta"),
                     "oof_ci90": o.get("ci90"), "our_test": t.get("macro_f1"),
                     "our_test_delta": t.get("delta")})
display(pd.DataFrame(rows).set_index(["arm", "target"]))

oof  oof_delta            oof_ci90  \
arm                      target                                            
+knn_in_category         category  0.8009     0.0241    [-0.0093, 0.091]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
+m1_in_category          category  0.8034     0.0266   [-0.0011, 0.0633]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
+unreviewed_augmentation category  0.8818     0.1050    [0.0388, 0.1869]   
                         priority  0.9265     0.0229       [0.0, 0.0717]   
-augmentation            category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m1                      category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2knn                   category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2lr                    category  0.7787     0.0019   [-0.0028, 0.0108]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2proto                 category  0.5611    -0.2158   [-0.3428, -0.088]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-urgency_features        category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.5880    -0.3156  [-0.4648, -0.1785]   
-wrapper_stripping       category  0.7316    -0.0452   [-0.1279, 0.0261]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
full                     category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
hard_targets             category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
mask_deadline            category  0.7768     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
proto_learned_only       category  0.5965    -0.1803  [-0.3054, -0.0575]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
proto_office_only        category  0.7812     0.0043   [-0.1268, 0.1088]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
tuned_stage_b            category  0.8280     0.0512        [0.0, 0.119]   
                         priority  0.9036     0.0000          [0.0, 0.0]   

                                   our_test  our_test_delta  
arm                      target                              
+knn_in_category         category    0.8113         -0.0442  
                         priority    0.8473          0.0000  
+m1_in_category          category    0.8467         -0.0088  
                         priority    0.8473          0.0000  
+unreviewed_augmentation category    0.8684          0.0128  
                         priority    0.9081          0.0607  
-augmentation            category    0.8556          0.0000  
                         priority    0.8473          0.0000  
-m1                      category    0.8556          0.0000  
                         priority    0.8473          0.0000  
-m2knn                   category    0.8556          0.0000  
                         priority    0.8473          0.0000  
-m2lr                    category    0.8492         -0.0064  
                         priority    0.8473          0.0000  
-m2proto                 category    0.6742         -0.1813  
                         priority    0.8473          0.0000  
-urgency_features        category    0.8556          0.0000  
                         priority    0.5985         -0.2489  
-wrapper_stripping       category    0.8351         -0.0204  
                         priority    0.7806         -0.0667  
full                     category    0.8556          0.0000  
                         pr